In [1]:
%pip install transformers bitsandbytes accelerate torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

CUDA available: True
GPU name: NVIDIA H200 NVL
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from larg

In [2]:
import os
print(os.getpid())

131


In [4]:
import os

folder_path = "tables/"

folder_path_LLM_statements = "LLAMA/b.LLM_Inferences"

folder_path_python_code = "LLAMA/c.checking_statements"

folder_path_python_output_checking_statements = "LLAMA/d.checking_statements_output"

Initialize model

In [5]:
#initalizing the model with 4 bit quantization

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "meta-llama/Llama-3.1-70B-Instruct"

# Configure 4-bit quantization to save VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [6]:
from transformers import pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [11]:
import re

def generate(b):
    
    # List all CSV files in the folder
    batch_start = b
    csv_files = [f for f in os.listdir(folder_path) 
                 if f.startswith("table_") and f.endswith(".csv") 
                 and batch_start <= int(f.split('_')[1].split('.')[0]) < batch_start + 10]
    csv_files.sort()  # optional: ensure consistent order
    
    print(csv_files)
    
    for csv_file in csv_files:
        full_path = os.path.join(folder_path, csv_file)
    
        # Read the table
        with open(full_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
    
        if not lines:
            print(f"{csv_file} is empty, skipping")
            continue
    
        # Extract header and rows
        header = lines[0]
        rows = lines[1:]
    
        print(f"Processing {csv_file}: {len(rows)} rows")
    
        prompt1 = [
        {"role": "system", "content": "You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables."},
        {"role": "user", "content": f"""
            Your task is to generate 10-20 natural language statements that describe patterns, relationships, and notable observations in this data. Generate as many **distinct, non-redundant** statements as the data supports.
    
    REQUIREMENTS:
    
    1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.
    2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21–43" or "high-income individuals"). Avoid arbitrary correlations.
    3. **Clarity**: Use natural language that is easy to understand. Avoid overly complex nested conditions. Keep conditions to 2–3 attributes max.
    4. **Variety**: Mix different types of statements:
       - Universal claims: "All X satisfy property Y"
       - Conditional claims: "If a person is X, then Y"
       - Existential claims: "There exists at least one X such that Y" — only use these when the combination of X and Y is semantically surprising or noteworthy
       - Majority/frequency claims: "Most X have property Y"
    
    NON-REDUNDANCY RULE — strictly enforce this:
    Before adding a statement, check whether a stricter or more informative version of it is also true. If "All X with A >= 10 have B >= 5" is true, do NOT also include "All X with A >= 10 have B >= 4" — only keep the tightest bound that still holds. Similarly, do not include a statement whose information is fully contained in another statement you've already written.
    
    AVOID:
    - Overly specific conditions that apply to only 1–2 individuals
    - Statements that express the same fact at different levels of precision (keep only the most precise version)
    - Bare existentials with no relational insight (e.g., "There exists a student with GPA 3.4" — this is not interesting on its own)
    - Probabilistic language (e.g., "likely", "probably") unless you have strong statistical support
    
    GOOD EXAMPLES (use these as a guide for style and depth):
    - For all individuals in the table, if the person is a woman, then their age is between 21 and 43 years.
    - There exists at least one man in the table whose resting heart rate is less than 70 bpm.
    - For all individuals with BMI greater than 30, their annual income is less than or equal to $45k.
    - If a person is a man aged over 50, then their BMI is less than 33.
    - All individuals who sleep less than 7 hours per night have a resting HR greater than 70 bpm.
    - Most individuals in the table have an average step count greater than 5000 steps per day.
    - If an individual is a woman aged below 30, then their height is greater than 150 cm.
    - Every individual with a BMI between 20 and 25 has an age less than 40.
    - For all individuals with annual income greater than $70k, their age is less than 50.
    
    BAD EXAMPLES (do not produce statements like these):
    - "There exists at least one individual with a BMI less than 20." — no meaningful relationship
    - "There exists at least one student whose study hours per week are less than 18." — no contrast or condition
    - "There exists at least one student whose age is 17 or less." — trivially true, no insight
    - "There exists at least one student whose extracurricular count is 5." — bare existential, not useful
    
    Here is the data:
    {lines}
    
    Generate your statements below, one per line. Write as many as the data supports, but stop before adding any statement that is redundant with one you've already written."""},]
        
        #Generate the statements necessary
        generation = generator(
        prompt1,
        do_sample=False,
        temperature=1.0,
        top_p=1,
        max_new_tokens=10000,
        eos_token_id=tokenizer.eos_token_id)
        # print(f"Generation: {generation[0]['generated_text']}")
        # Get the assistant message from generated_text
        statements_LLM_output = generation[-1]['generated_text'][-1]  # last item
        clean_statements_LLM_output_text = statements_LLM_output['content']  # this is your CSV string
        statements_LLM_output_text = re.sub(r"<think>.*?</think>", "", clean_statements_LLM_output_text, flags=re.DOTALL)
        # Preview
        # print(statements_LLM_output_text[1000:2000])
        #save the output of the LLM generated tasks
        file_name_LLM = f"LLM_statements_{csv_file[:-4]}.txt"
    
        folder_path_LLM_statements = "LLAMA/b.LLM_Inferences"
    
        full_path_LLM_statements = os.path.join(folder_path_LLM_statements, file_name_LLM)
    
    
        with open(full_path_LLM_statements, "w", encoding="utf-8") as f:
          f.write(statements_LLM_output_text)
        print(f"Saved {file_name_LLM}.")


In [13]:
import re
import subprocess
import pandas as pd

# helper to extract
def extract_python_code(raw: str) -> str:
    # Match ```python ... ``` or ``` ... ``` (non-greedy, dotall)
    match = re.search(r"```(?:python)?\s*\n(.*?)```", raw, re.DOTALL)
    if match:
        return match.group(1).strip()
    # No fences found — return as-is (model already output plain code)
    return raw.strip()

LLM_statement_text_files = sorted(
    f for f in os.listdir(folder_path_LLM_statements) if f.endswith(".txt")
)

for LLM_statement_text_file in LLM_statement_text_files:
    full_path_stmt = os.path.join(folder_path_LLM_statements, LLM_statement_text_file)

    with open(full_path_stmt, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{LLM_statement_text_file} is empty, skipping")
        continue

    print(f"\nProcessing {LLM_statement_text_file}.")

    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    for csv_file in csv_files:
        if csv_file[:-4] not in LLM_statement_text_file:
            continue

        full_csv_path = os.path.join(folder_path, csv_file)
        df = pd.read_csv(full_csv_path)

        prompt2 = [
            {"role": "system", "content": "You are an expert data analyst."},
            {"role": "user", "content": f"""Do NOT repeat the instructions or the code provided.
Only output the requested Python code. Do NOT wrap the code in markdown fences.
Here's your task: Given the following statements {lines} and the header names of the table {df.head(0)}, write a python code (using pandas package)
that checks whether each statement is True or False and prints a justification.
The CSV is already located at: "{full_csv_path}" — hardcode this path directly in the script (no sys.argv).
It should also convert any turn numbers stored as strings into integers.
Everything you output must be valid, immediately runnable Python with no markdown or commentary outside comments.

Here is an example structure to follow:
import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\\nStatement {{statement_no}}: {{status}}")
    print(f"  - {{description}}")
    print(f"  - Explanation: {{explanation}}")

def stmt_1(df: pd.DataFrame):
    \"\"\"1. For all individuals, if the person is a woman, then her age is between 21 and 43.\"\"\"
    women = df[df["gender"] == "F"]
    condition = women["age"].between(21, 43, inclusive="both")
    truth = condition.all()
    if truth:
        expl = f"All {{len(women)}} women are aged 21–43."
    else:
        viol = women[~condition]
        expl = f"{{len(viol)}} women violate the rule (ages: {{', '.join(map(str, viol['age'].tolist()))}})."
    return truth, expl

def main():
    df = pd.read_csv("{full_csv_path}")
    checks = [(1, stmt_1)]  # extend for all statements
    for num, func in checks:
        truth, explanation = func(df)
        print_result(num, func.__doc__.strip(), truth, explanation)

if __name__ == "__main__":
    main()"""},
        ]

        # ── Generate ───────────────────────────────────────────────────────────
        generation = generator(
            prompt2,
            do_sample=False,
            temperature=1.0,
            top_p=1,
            max_new_tokens=100000,
            eos_token_id=tokenizer.eos_token_id,
        )

        python_LLM_output = generation[-1]["generated_text"][-1]
        raw_code = python_LLM_output["content"]

        # ── Clean: strip markdown fences reliably ──────────────────────────────
        python_code = extract_python_code(raw_code)

        # ── Save ───────────────────────────────────────────────────────────────
        python_file_name_LLM = f"python_code_{csv_file[:-4]}.py"
        full_path_py = os.path.join(folder_path_python_code, python_file_name_LLM)

        with open(full_path_py, "w", encoding="utf-8") as f:
            f.write(python_code)
        print(f"Saved {python_file_name_LLM}")

        # ── Execute automatically ──────────────────────────────────────────────
        print(f"Running {python_file_name_LLM}...")
        result = subprocess.run(
            ["python3", full_path_py],
            capture_output=True,
            text=True,
        )

        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print(f"[ERROR] Script exited with code {result.returncode}")
            print(result.stderr)
        else:
            print(f"[OK] {python_file_name_LLM} completed successfully.")

        results_file_name = f"validation_llama_inferences_{csv_file[:-4]}.txt"
        full_path_results_file = os.path.join(folder_path_python_output_checking_statements, results_file_name)

        with open(full_path_results_file, "w", encoding="utf-8") as f:
            f.write(result.stdout)
            if result.returncode != 0:
                f.write(f"\n[ERROR] Script exited with code {result.returncode}\n")
                f.write(result.stderr)

        print(f"Saved {results_file_name}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Processing LLM_statements_table_0.txt.


KeyboardInterrupt: 

In [ ]:
import re

batchs = [20, 30, 40, 50, 60, 70, 80, 90]

for b in batchs:
    generate(b)

[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['table_20.csv', 'table_21.csv', 'table_22.csv', 'table_23.csv', 'table_24.csv', 'table_25.csv', 'table_26.csv', 'table_27.csv', 'table_28.csv', 'table_29.csv']
Processing table_20.csv: 15 rows


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved LLM_statements_table_20.txt.
Processing table_21.csv: 15 rows


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Saved LLM_statements_table_30.txt.
Processing table_31.csv: 15 rows


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved LLM_statements_table_31.txt.
Processing table_32.csv: 15 rows


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved LLM_statements_table_32.txt.
Processing table_33.csv: 15 rows
